In [4]:
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
! pip install langchain_community

- !pip install -q langchain
→ Framework to connect LLMs, embeddings, and tools into a pipeline.

- !pip install -q torch
→ Runs the deep learning models (backend engine).

!pip install -q transformers
→ Provides pre-trained language models and tokenizers from Hugging Face.

!pip install -q sentence-transformers
→ Generates vector embeddings that capture text meaning.

!pip install -q datasets
→ Loads ready-to-use NLP datasets from Hugging Face.

!pip install -q faiss-cpu
→ Creates a vector database to search similar text chunks quickly.

!pip install langchain_community
→ Adds external data connectors and integrations for Langchain.

In [5]:
# 2- Load the dataset from Hugging Face using Langchain

# Import the HuggingFaceDatasetLoader from Langchain
from langchain.document_loaders import HuggingFaceDatasetLoader

# Defining the name of the dataset we want to use (this one contains sample question-answering data)
dataset_name = "databricks/databricks-dolly-15k"

page_content_column = "context" # Tells the loader which column in the dataset contains the main content (context text)

# Creating a dataset loader instance with the dataset name and the content column
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)

# Loading the dataset — this pulls in the documents from Hugging Face
data = loader.load()

print(data[:2]) # Prints the first two entries in the dataset to check what it looks like

[Document(metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}, page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia\'s domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."'), Document(metadata={'instruction': 'Which is a species of fish? Tope or Rope', 'response': 'Tope', 'category': 'classification'}, page_content='""')]


In [6]:
# 3. Split the documents (into smaller chunks):

# Import the text splitter from Langchain
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Creating an instance of the splitter
# chunk_size: maximum number of characters in each chunk
# chunk_overlap: number of characters that overlap between chunks (helps preserve context)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Applying the splitter to the loaded dataset to break it into smaller pieces
docs = text_splitter.split_documents(data)

# (Optional) Printing the first chunk to see the result
print(docs[0])

page_content='"Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney."' metadata={'instruction': 'When did Virgin Australia start operating?', 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [8]:
# 4. Embed the text:
# Generating text embeddings using sentence-transformers

# Import the embedding class from Langchain
from langchain.embeddings import HuggingFaceEmbeddings

# Defining the path to the pre-trained embedding model
# This model turns each chunk of text into a vector (a list of numbers that represents meaning)
modelPath = "sentence-transformers/all-MiniLM-L6-v2"

# Set model configuration options
# 'device': 'cpu' tells the model to run on CPU (can be changed to 'cuda' for GPU)
model_kwargs = {'device': 'cpu'}

# Set encoding options
# normalize_embeddings=False means we keep raw embeddings (some applications normalize them)
encode_kwargs = {'normalize_embeddings': False}

# Create the embedding model using HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# (Optional) Test embedding creation by generating an embedding for a test sentence
text = "This is a test document."
query_result = embeddings.embed_query(text)

# Print first few values from the embedding vector to verify
print(query_result[:3])

[-0.038338541984558105, 0.12346471846103668, -0.02864297851920128]


small explanation of 4
- Each chunk of text becomes a vector, a list of numbers that captures the meaning of the chunk.
- These embeddings allow us to later compare texts based on their meaning, not just matching keywords.
- The model "all-MiniLM-L6-v2" is small and fast, but performs well for semantic similarity tasks.

In [9]:
# 5. Create a vector store: (using FAISS)

# Import FAISS from Langchain's vectorstores module
from langchain.vectorstores import FAISS

# Create a FAISS vector store using the document chunks and their embeddings
# This will allow fast similarity search later when retrieving relevant documents
db = FAISS.from_documents(docs, embeddings)

small explanation of 5:
- FAISS (Facebook AI Similarity Search) is a special kind of database designed to search through vectors quickly.
- we now have a database where we can ask:
“Find me the chunks most similar to this question,”
and it will return the closest matches based on meaning, not exact words.
- This is what powers the retrieval part of the RAG system.

In [10]:
# 6. Prepare the LLM model (Question Answering Language Model)

# Import the required modules from Hugging Face and Langchain
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain import HuggingFacePipeline

# Define the model name (a small and efficient question answering model)
model_name = "Intel/dynamic_tinybert"

# Load the tokenizer — it prepares the text input for the model
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)

# Load the pre-trained question answering model
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Create a Hugging Face pipeline for question answering
# This wraps the model and tokenizer into a single callable interface
qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    return_tensors='pt'
)

# Wrap the Hugging Face pipeline into a Langchain-compatible LLM
# model_kwargs allow you to control generation behavior
llm = HuggingFacePipeline(
    pipeline=qa_pipeline,
    model_kwargs={"temperature": 0.7, "max_length": 512}
)

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Invalid model-index. Not loading eval results into CardData.
Device set to use cpu
<ipython-input-10-e57937ed245d>:27: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(


for 6:
- we are now preparing the language model (LLM) that will generate answers.
- The TinyBERT model is small, fast, and trained specifically for question answering tasks.
- The model reads a question and a context, then tries to extract the best answer from the context.
- Wrapping it in a pipeline and then in a Langchain wrapper allows it to work inside your RAG system easily.

In [11]:
# 7. Build the Retrieval QA Chain:

# Import the RetrievalQA class from Langchain
from langchain.chains import RetrievalQA

# Create a retriever from the FAISS vector store
# The retriever will return the top-k most similar document chunks based on a user's question
retriever = db.as_retriever(search_kwargs={"k": 4})  # 'k' is the number of documents to retrieve

# Build the RetrievalQA chain
# This combines the retriever with the LLM model so the system first retrieves relevant content,
# then uses the LLM to generate an answer from that content
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",  # You could also use "stuff" or "map_reduce" depending on use case
    retriever=retriever,
    return_source_documents=False  # Set to True if you want to see which documents were used to answer
)

for 7:
- The Retriever searches the vector store and finds the most relevant chunks of text.
- The LLM model then reads those chunks and answers your question.
- The chain type "refine" means it reads documents one by one and improves the answer step by step.
- Now we have a complete Retrieval Augmented Generation (RAG) system ready to use.

In [13]:
# 8. Test your RAG system (Ask a question and get an answer from RAG system)

# Define a question that you want to ask
question = "What is cheesemaking?"

# Run the RetrievalQA chain with your question
# The system will:
# 1. Search the most relevant chunks from the dataset
# 2. Use the LLM to answer the question based on those chunks
# result = qa.run({"query": question}) **********qa.run didn't work
result = qa.invoke({"query": question})

# Print the result — the final answer generated by your RAG system
print(result)

/usr/local/lib/python3.11/dist-packages/transformers/pipelines/question_answering.py:391: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(


ValueError: Context information is below. 
------------
"The goal of cheese making is to control the spoiling of milk into cheese. The milk is traditionally from a cow, goat, sheep or buffalo, although, in theory, cheese could be made from the milk of any mammal. Cow's milk is most commonly used worldwide. The cheesemaker's goal is a consistent product with specific characteristics (appearance, aroma, taste, texture). The process used to make a Camembert will be similar to, but not quite the same as, that used to make Cheddar.\n\nSome cheeses may be deliberately left to ferment from naturally airborne spores and bacteria; this approach generally leads to a less consistent product but one that is valuable in a niche market.\n\nCulturing\nCheese is made by bringing milk (possibly pasteurised) in the cheese vat to a temperature required to promote the growth of the bacteria that feed on lactose and thus ferment the lactose into lactic acid. These bacteria in the milk may be wild, as is th

for8:
- When we ask a question, our RAG system searches the vector store for the most relevant information.
- Then it passes that information to the LLM, which generates the answer based on that content.
- The final printed output is the answer generated by the model, grounded in your dataset.